# CIBC — Digital Insurance Platform Acquisition
## Monte Carlo Simulation — R1: Data Privacy Breach

**ALY6130 | Group 7 | June 2026**

This notebook contains only the Monte Carlo simulation for R1 (Data Privacy Breach).
Full analysis including Sensitivity Analysis (R2), EMV (R3), and ML models is in:
`quantitative_analysis.ipynb`

---
**Risk:** R1 — Data Privacy Breach | Likelihood = 9 | Impact = 8 | Score = 72 | Priority: HIGH

**Method:** Monte Carlo Simulation — Triangular Distribution — 10,000 Iterations

**Data source:** Proxy and synthetic datasets derived from the Group 7 Risk Register,
Risk Calculation Sheet, and scenario-based cost assumptions.
See: `Module4_Data_Source_Risk_Register_and_Calculations.xlsx`

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SETUP — Import libraries
# ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import triang
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)   # reproducibility
N = 10000            # Monte Carlo iterations

print('Libraries loaded. Monte Carlo iterations:', N)

---
## PART 3 — R1: Monte Carlo Simulation (Data Privacy Breach)

**Method:** Triangular distribution modeling total financial loss across 10,000 simulated scenarios.

**Cost inputs from CIBC risk register:**
- Optimistic: CAD $25M (PIPEDA fine floor + minimal class-action)
- Most Likely: CAD $75M (mid-range fine + class-action + remediation)
- Pessimistic: CAD $150M (maximum fine + full class-action + 6–12 month shutdown)

**I&W Connection:** The Monte Carlo output quantifies the financial consequence distribution that the I&W warning thresholds are designed to prevent.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# R1 — MONTE CARLO SIMULATION
# Triangular distribution: Optimistic=$25M, MostLikely=$75M, Pessimistic=$150M
# Cost inputs sourced directly from Group 7 risk register R1 consequence field
# ─────────────────────────────────────────────────────────────────

r1_opt  = 25    # CAD $M — optimistic (PIPEDA fine floor)
r1_ml   = 75    # CAD $M — most likely (mid-range scenario)
r1_pess = 150   # CAD $M — pessimistic (full class-action + shutdown)

# Triangular distribution shape parameter
c = (r1_ml - r1_opt) / (r1_pess - r1_opt)

# Run simulation
r1_losses = triang.rvs(c, loc=r1_opt, scale=r1_pess - r1_opt, size=N)

# Summary statistics
r1_mean  = np.mean(r1_losses)
r1_p50   = np.percentile(r1_losses, 50)
r1_p90   = np.percentile(r1_losses, 90)
r1_p95   = np.percentile(r1_losses, 95)
r1_p99   = np.percentile(r1_losses, 99)
r1_prob_over_100 = np.mean(r1_losses > 100) * 100
r1_prob_over_125 = np.mean(r1_losses > 125) * 100

print('R1 — Data Privacy Breach: Monte Carlo Results')
print('=' * 50)
print(f'  Simulation runs:          {N:,}')
print(f'  Input — Optimistic:       CAD ${r1_opt}M')
print(f'  Input — Most Likely:      CAD ${r1_ml}M')
print(f'  Input — Pessimistic:      CAD ${r1_pess}M')
print(f'  Mean expected loss:       CAD ${r1_mean:.1f}M')
print(f'  Median (50th pct):        CAD ${r1_p50:.1f}M')
print(f'  90th percentile:          CAD ${r1_p90:.1f}M')
print(f'  95th percentile:          CAD ${r1_p95:.1f}M')
print(f'  99th percentile:          CAD ${r1_p99:.1f}M')
print(f'  P(Loss > CAD $100M):      {r1_prob_over_100:.1f}%')
print(f'  P(Loss > CAD $125M):      {r1_prob_over_125:.1f}%')
print()
print(f'  >> Recommended contingency reserve: CAD ${r1_p90:.0f}M (90th percentile)')
print(f'  >> Board should stress-test to:      CAD ${r1_p95:.0f}M (95th percentile)')

In [ ]:
# ─── R1 CHART ───────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.hist(r1_losses, bins=80, color='#D42020', alpha=0.75, edgecolor='#8B0000', linewidth=0.4)
ax1.axvline(r1_mean, color='#1F3864', lw=2.0, ls='--', label=f'Mean = CAD ${r1_mean:.1f}M')
ax1.axvline(r1_p90,  color='#FF6600', lw=2.0, ls='-',  label=f'90th Pct = CAD ${r1_p90:.1f}M  ← Recommended Reserve')
ax1.axvline(r1_p95,  color='#8B0000', lw=2.0, ls=':',  label=f'95th Pct = CAD ${r1_p95:.1f}M  ← Board Stress Test')
ax1.set_xlabel('Total Financial Loss (CAD $M)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Frequency (out of 10,000 simulations)', fontsize=11, fontweight='bold')
ax1.set_title('R1 — Data Privacy Breach\nMonte Carlo Simulation: Total Financial Loss Distribution (10,000 iterations)',
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.text(0.97, 0.70,
    f'Optimistic:  CAD ${r1_opt}M\nMost Likely: CAD ${r1_ml}M\nPessimistic: CAD ${r1_pess}M\n'
    f'P(Loss > $100M): {r1_prob_over_100:.1f}%',
    transform=ax1.transAxes, ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='#FFF3F3', edgecolor='#D42020', alpha=0.9))
fig1.text(0.01, -0.04,
    'Figure 1. Monte Carlo simulation — R1 Data Privacy Breach. '
    'Triangular distribution inputs: Optimistic=CAD $25M (PIPEDA fine floor), '
    'Most Likely=CAD $75M (mid-range fine + class-action + remediation), '
    'Pessimistic=CAD $150M (maximum fine + full class-action + 6-12 month shutdown). '
    'Source: Group 7 Risk Register, ALY6130, Spring 2026.',
    fontsize=7, color='#555555', style='italic')
fig1.subplots_adjust(bottom=0.15)
plt.savefig('fig1_r1_montecarlo.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 1 saved: fig1_r1_montecarlo.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MONTE CARLO RESULTS SUMMARY
# Key outputs used in Signature Assessment report (Section 4.2)
# ─────────────────────────────────────────────────────────────────

print("=" * 55)
print("R1 — Data Privacy Breach: Monte Carlo Summary")
print("=" * 55)
print(f"  Iterations run:          10,000")
print(f"  Distribution:            Triangular")
print(f"  Optimistic input:        CAD $25M")
print(f"  Most Likely input:       CAD $75M")
print(f"  Pessimistic input:       CAD $150M")
print()
print(f"  Mean expected loss:      CAD ${r1_mean:.1f}M")
print(f"  Median (50th pct):       CAD ${r1_p50:.1f}M")
print(f"  90th percentile:         CAD ${r1_p90:.1f}M  ← Recommended contingency reserve")
print(f"  95th percentile:         CAD ${r1_p95:.1f}M  ← Board stress-test benchmark")
print(f"  P(Loss > CAD $100M):     {r1_prob_over_100:.1f}%")
print()
print("RT&RP Update:")
print(f"  Contingency reserve set at CAD ${r1_p90:.0f}M (90th percentile)")
print(f"  Board stress-test at:        CAD ${r1_p95:.0f}M (95th percentile)")
print(f"  P(loss > $100M) = {r1_prob_over_100:.1f}% confirms R1 is quantitatively material")
print("=" * 55)
